In [2]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

import gc, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForSequenceClassification
from peft import PeftModel
import bitsandbytes as bnb

In [ ]:
# IDs
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
finetuned_repo_id_4bit = "eduhuemar001/tinyllama-german-sentiment-4bit"
finetuned_repo_id_8bit = "eduhuemar001/tinyllama-german-sentiment-8bit"

# Helper functions
def print_gpu_mem(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA not available"); return 0
    torch.cuda.synchronize()
    a = torch.cuda.memory_allocated()
    print(f"[{tag}] allocated={a/1e9:.3f} GB")
    return a

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def load_fp16_model(model_id: str):
    # Try'dtype' or 'torch_dtype'
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.float16,
            device_map="cuda" if torch.cuda.is_available() else None,
        )
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="cuda" if torch.cuda.is_available() else None,
        )

# fp16 base
free_gpu()
baseline = print_gpu_mem("before fp16")
base_fp16 = load_fp16_model(base_model_id)
base_fp16.eval()
alloc_after = print_gpu_mem("after  fp16")
delta_fp16 = (alloc_after - baseline) / 1e9
del base_fp16; free_gpu(); print_gpu_mem("after unload fp16")

# 4-bit fine-tuned model
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

baseline = print_gpu_mem("before 4-bit finetuned")
try:
    base_4bit = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=quant_config_4bit, device_map="auto",
    )
    model_4bit = PeftModel.from_pretrained(base_4bit, finetuned_repo_id_4bit)
except Exception:
    model_4bit = AutoModelForCausalLM.from_pretrained(
        finetuned_repo_id_4bit, quantization_config=quant_config_4bit, device_map="auto",
    )
model_4bit.eval(); model_4bit.config.use_cache = True
alloc_after = print_gpu_mem("after  4-bit finetuned")
delta_4bit = (alloc_after - baseline) / 1e9

# cleanup before 8-bit section
try: del model_4bit
except: pass
try: del base_4bit
except: pass
free_gpu(); baseline = print_gpu_mem("after unload 4-bit")

# 8-bit fine-tuned model
quant8 = BitsAndBytesConfig(load_in_8bit=True)

print_gpu_mem("before 8-bit finetuned")
baseline = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
try:
    # repo contains LoRA adapters
    base_8bit = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=quant8, device_map="auto",
    )
    model_8bit = PeftModel.from_pretrained(base_8bit, finetuned_repo_id_8bit)
except Exception:
    # repo is a merged fine-tuned model
    model_8bit = AutoModelForCausalLM.from_pretrained(
        finetuned_repo_id_8bit, quantization_config=quant8, device_map="auto",
    )

model_8bit.eval(); model_8bit.config.use_cache = True
alloc_after = print_gpu_mem("after  8-bit finetuned")
delta_8bit = (alloc_after - baseline) / 1e9

# verify loaded in 8-bit
is_8bit_flag = getattr(model_8bit, "is_loaded_in_8bit", False)
linear8_count = sum(1 for m in model_8bit.modules() if isinstance(m, bnb.nn.Linear8bitLt))
print(f"is_loaded_in_8bit: {is_8bit_flag}")

print("\n")
print(f"fp16 base        ≈ {delta_fp16:.3f} GB")
print(f"4-bit finetuned  ≈ {delta_4bit:.3f} GB")
print(f"8-bit finetuned  ≈ {delta_8bit:.3f} GB")

[before fp16] allocated=0.000 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

In [ ]:
bert_model_id = "oliverguhr/german-sentiment-bert"

def model_param_size_mb(model_id: str) -> float:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, torch_dtype=torch.float32, device_map=None
    )
    total_bytes = 0
    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()
    return total_bytes / (1024 ** 2)

size_mb = model_param_size_mb(bert_model_id)
print(f"Parameter size (FP32): {size_mb:.1f} MB")